# Домашнее задание № 10. Машинный перевод

## Задание 1 (6 баллов + 2 доп балла).
Попробуйте обучить трансформерную переводную модель также как в семинаре, но вместо пары языков возьмите несколько исходных языков сразу и совместите их в одной модели. Например, создайте модель EN, FR, DE -> RU (можно использовать другие языки).
Для того, чтобы обучить такую модель вам нужно будет собрать датасет из нескольких парных корпусов. Можете просто склеить все в одну большую выборку, а можете взять какую-то выборку из единичных корпусов, чтобы общее количество данных не было таким большим.
Используйте как минимум 3 исходных языка.
Не забудьте переобучить токенизаторы на новых датасетах!

Можно использовать как основу любую из реализаций из семинара (MultiHeadAttention, nn.Transformer, global + local attention).
Параметры ниже точно работают в колабе и модель обучается достаточно быстро. Попробуйте их немного увеличить (batch size возможно придется наоборот уменьшить). Обучайте модель хотя бы 5 эпох, а желательно больше, чтобы тестовые примеры начали переводиться более менее адекватно.

После обучения возьмите хотя бы по 50 примеров на язык из тестовой части корпуса и переведите их. Оцените качество переводов с помощью метрики BLEU и ChrF для каждого из языков.
Обязательно посмотрите на получаемые переводы! Если все переводы это пустые строки или зацикленные повторы одного символа, то пообучайте модель подольше.
Найдите лучшие (как минимум по 2 на язык) переводы согласно этими метрикам и проверьте действительно ли они хорошие.
В качестве дополнительного теста проверьте может ли модель перевести пару предложений на каком-то другом хотя бы немного близком языке (например, если в обучающих исходных текстах вы использовали EN, FR, IT то попробуйте переводить испанский). Получаемый результат совсем неправильный или модель смогла найти и использовать какие-то общие закономерности?



Чтобы получить 2 доп балла вам нужно будет придумать как оптимизировать функцию translate. Сейчас она работает только с одним текстом - это не эффективно. Можно генерировать переводы сразу для нескольких текстов (батча). Главная сложность с таким подходом состоит в том, что генерируемые тексты будут заканчиваться в разное время и нужно сделать столько итераций, сколько нужно для завершения всех текстов (т.е. условие на то, что последний токен не равен [EOS] в текущем коде не сработает).
ВАЖНО - недостаточно просто изменить входной аргумент с text на texts и добавить еще один цикл по texts! Сама модель должна вызываться на нескольких текстах! Функция с batch prediction должна работать быстрее, поэтому переведите всю тестовую выборку и оцените качество BLEU как минимум на 2000 примеров (а лучше на всем тестовом корпусе).

In [1]:
!pip3 install "datasets<4.0.0" sacrebleu torchtune torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
import torch
import torch.nn as nn
import torch.utils.data
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from tokenizers import decoders
from datasets import load_dataset
import numpy as np
from sklearn.model_selection import train_test_split
import sacrebleu
import random
import os
from torchtune.modules import RotaryPositionalEmbeddings
from torch.nn import Transformer
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### 1. Загружаем и готовим данные

Я обнаружил, что в используемом в семинаре Opus как будто бы нет других параллельных пар, ггде исходный язык — неанглийский. Поэтому я воспользовался датасетом Tatoeba через HF Datasets. Там тоже не всё гладко, и, кажется, новые версии библиотеки его  в таком виде уже не поддерживают, поэтому выше и был даунгрейт.

В качестве исходных языков возьмём английский, немецкий и испанский — два германских и один романский. А в качестве языков для тестирования в zero-shot-сетапе — нидерландский (германский) и французский (романский). Гипотеза состоит в том, что нидерландский будет переводится лучше, потому что в нашей обучающей выборке больше данных германских языков, чем романских.

In [ ]:
def load_multilingual_data(lang_pair, fraction=1.0, test_size=5000, seed=42):
    # С помощью fraction можно контролировать размер выборки
    l1, l2 = lang_pair.split('-')
    print(f"Loading {lang_pair}...")
    dataset = load_dataset(
        "tatoeba",
        lang1=l1,
        lang2=l2,
        split="train",
        trust_remote_code=True)

    # Извлекаем тексты
    src_lines = [item[l1] for item in dataset['translation']]
    tgt_lines = [item[l2] for item in dataset['translation']]

    if fraction < 1.0:
        random.seed(seed)
        combined = list(zip(src_lines, tgt_lines))
        random.shuffle(combined)
        limit = int(len(combined) * fraction)
        sampled = combined[:limit]
        src_lines, tgt_lines = zip(*sampled)

    actual_test_size = min(test_size, int(len(src_lines) * 0.1))

    # Разбиваем на train и test
    src_train, src_test, tgt_train, tgt_test = train_test_split(
        src_lines, tgt_lines, test_size=actual_test_size, random_state=seed
    )

    return list(src_train), list(tgt_train), list(src_test), list(tgt_test)

In [4]:
fraction = 1.0

# Обучающая выборка
en_train, ru_train_en, en_test, ru_test_en = load_multilingual_data(
    'en-ru', fraction)
de_train, ru_train_de, de_test, ru_test_de = load_multilingual_data(
    'de-ru', fraction)
es_train, ru_train_es, es_test, ru_test_es = load_multilingual_data(
    'es-ru', fraction)

# Склеиваем обучающую выборку из 3 языков
src_train_all = en_train + de_train + es_train
ru_train_all = ru_train_en + ru_train_de + ru_train_es

# Тестовые примеры для zero-shot-сетапа
_, _, nl_test, ru_test_nl = load_multilingual_data(
    'nl-ru', fraction=0.1, test_size=500)
_, _, fr_test, ru_test_fr = load_multilingual_data(
    'fr-ru', fraction=0.1, test_size=500)

print(f"Итого обучающих примеров: {len(src_train_all)}")

with open('src_train_combined.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(src_train_all))
with open('ru_train_combined.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(ru_train_all))

Loading en-ru...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

tatoeba.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Loading nl-ru...


Generating train split: 0 examples [00:00, ? examples/s]

Loading fr-ru...


Generating train split:   0%|          | 0/195161 [00:00<?, ? examples/s]

Итого обучающих примеров: 719182


### 2. Обучаем токенизатор

In [5]:
tokenizer_src = Tokenizer(BPE())
tokenizer_src.pre_tokenizer = Whitespace()
trainer_src = BpeTrainer(special_tokens=["[PAD]"], end_of_word_suffix='</w>')
tokenizer_src.train(files=["src_train_combined.txt"], trainer=trainer_src)

tokenizer_ru = Tokenizer(BPE())
tokenizer_ru.pre_tokenizer = Whitespace()
trainer_ru = BpeTrainer(
    special_tokens=[
        "[PAD]",
        "[BOS]",
        "[EOS]"],
    end_of_word_suffix='</w>')
tokenizer_ru.train(files=["ru_train_combined.txt"], trainer=trainer_ru)

tokenizer_src.decoder = decoders.BPEDecoder(suffix='</w>')
tokenizer_ru.decoder = decoders.BPEDecoder(suffix='</w>')

PAD_IDX = tokenizer_ru.token_to_id('[PAD]')

In [6]:
def encode(text, tokenizer, max_len, encoder=False):
    if encoder:
        return tokenizer.encode(text).ids[:max_len]
    else:
        return [tokenizer.token_to_id(
            '[BOS]')] + tokenizer.encode(text).ids[:max_len] + [tokenizer.token_to_id('[EOS]')]

### 3. Токенизированный датасет и даталоадеры

In [7]:
max_len_src, max_len_ru = 60, 60

X_src = [encode(t, tokenizer_src, max_len_src, encoder=True)
         for t in src_train_all]
X_ru = [encode(t, tokenizer_ru, max_len_ru) for t in ru_train_all]

In [8]:
class MTDataset(torch.utils.data.Dataset):
    def __init__(self, texts_src, texts_ru):
        self.texts_src = [torch.LongTensor(sent) for sent in texts_src]
        self.texts_src = torch.nn.utils.rnn.pad_sequence(
            self.texts_src, batch_first=True, padding_value=PAD_IDX)

        self.texts_ru = [torch.LongTensor(sent) for sent in texts_ru]
        self.texts_ru = torch.nn.utils.rnn.pad_sequence(
            self.texts_ru, batch_first=True, padding_value=PAD_IDX)
        self.length = len(texts_src)

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        return self.texts_src[index], self.texts_ru[index]

In [9]:
X_src_train, X_src_valid, X_ru_train, X_ru_valid = train_test_split(
    X_src, X_ru, test_size=0.02, random_state=42)

batch_size = 256
training_generator = torch.utils.data.DataLoader(
    MTDataset(
        X_src_train,
        X_ru_train),
    batch_size=batch_size,
    shuffle=True)
valid_generator = torch.utils.data.DataLoader(
    MTDataset(
        X_src_valid,
        X_ru_valid),
    batch_size=batch_size,
    shuffle=False)

### 4. Модель

Я взял самую простую реализацию из семинара, с готовым трансформерным блоком.

In [10]:
class TransformerEncoderDecoderTied(nn.Module):
    def __init__(
            self,
            vocab_size_enc,
            vocab_size_dec,
            d_model,
            num_heads,
            ff_dim,
            num_layers,
            dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.embedding_enc = nn.Embedding(vocab_size_enc, d_model)
        self.embedding_dec = nn.Embedding(vocab_size_dec, d_model)
        self.positional_encoding = RotaryPositionalEmbeddings(
            d_model // num_heads, max_seq_len=128)

        self.transformer = Transformer(
            d_model=d_model,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True)

        self.output_layer = nn.Linear(d_model, vocab_size_dec, bias=False)
        self.output_layer.weight = self.embedding_dec.weight

    def forward(
            self,
            src,
            tgt,
            src_key_padding_mask=None,
            tgt_key_padding_mask=None):
        src_embedded = self.embedding_enc(src)
        B, S, E = src_embedded.shape
        src_embedded = self.positional_encoding(
            src_embedded.view(
                B,
                S,
                self.num_heads,
                E //
                self.num_heads)).view(
            B,
            S,
            E)

        tgt_embedded = self.embedding_dec(tgt)
        B, T, E = tgt_embedded.shape
        tgt_embedded = self.positional_encoding(
            tgt_embedded.view(
                B,
                T,
                self.num_heads,
                E //
                self.num_heads)).view(
            B,
            T,
            E)

        tgt_mask = (
            ~torch.tril(
                torch.ones(
                    (T, T), dtype=torch.bool))).to(
            tgt.device)

        encoder_output = self.transformer.encoder(
            src_embedded, src_key_padding_mask=src_key_padding_mask)
        decoder_output = self.transformer.decoder(
            tgt_embedded,
            encoder_output,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask)
        return self.output_layer(decoder_output)

In [11]:
vocab_size_enc = tokenizer_src.get_vocab_size()
vocab_size_dec = tokenizer_ru.get_vocab_size()

embed_dim = 64
num_heads = 4
ff_dim = embed_dim * 2
num_layers = 2

model = TransformerEncoderDecoderTied(
    vocab_size_enc,
    vocab_size_dec,
    embed_dim,
    num_heads,
    ff_dim,
    num_layers).to(DEVICE)

### 5. Функции обучения

In [12]:
def train_epoch(model, iterator, optimizer, loss_fn, scheduler):
    epoch_loss = []
    model.train()
    pbar = tqdm(iterator, desc='Train', leave=False)
    for texts_src, texts_ru in pbar:
        texts_src, texts_ru = texts_src.to(DEVICE), texts_ru.to(DEVICE)
        texts_ru_input = texts_ru[:, :-1]
        texts_ru_out = texts_ru[:, 1:]

        src_padding_mask = (texts_src == PAD_IDX)
        tgt_padding_mask = (texts_ru_input == PAD_IDX)

        optimizer.zero_grad()
        logits = model(
            texts_src,
            texts_ru_input,
            src_padding_mask,
            tgt_padding_mask)
        B, S, C = logits.shape
        loss = loss_fn(logits.reshape(B * S, C), texts_ru_out.reshape(B * S))
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss.append(loss.item())
        pbar.set_postfix(loss=f'{np.mean(epoch_loss):.4f}')
    return np.mean(epoch_loss)


@torch.no_grad()
def evaluate(model, iterator, loss_fn):
    epoch_loss = []
    model.eval()
    for texts_src, texts_ru in tqdm(iterator, desc='Eval', leave=False):
        texts_src, texts_ru = texts_src.to(DEVICE), texts_ru.to(DEVICE)
        texts_ru_input = texts_ru[:, :-1]
        texts_ru_out = texts_ru[:, 1:]

        src_padding_mask = (texts_src == PAD_IDX)
        tgt_padding_mask = (texts_ru_input == PAD_IDX)

        logits = model(
            texts_src,
            texts_ru_input,
            src_padding_mask,
            tgt_padding_mask)
        B, S, C = logits.shape
        loss = loss_fn(logits.reshape(B * S, C), texts_ru_out.reshape(B * S))
        epoch_loss.append(loss.item())
    return np.mean(epoch_loss)

### 6. Обучаем

In [13]:
NUM_EPOCHS = 10
loss_fn = torch.nn.CrossEntropyLoss(
    ignore_index=PAD_IDX,
    label_smoothing=0.1).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=0.002, pct_start=0.10,
    steps_per_epoch=len(training_generator), epochs=NUM_EPOCHS
)

In [14]:
best_val_loss = float('inf')

for epoch in tqdm(range(1, NUM_EPOCHS + 1), desc='Epochs'):
    train_loss = train_epoch(
        model,
        training_generator,
        optimizer,
        loss_fn,
        scheduler)
    val_loss = evaluate(model, valid_generator, loss_fn)

    print(
        f"Epoch {epoch} | Train loss: {train_loss:.3f} | Val loss: {val_loss:.3f}")
    if val_loss < best_val_loss:
        torch.save(model.state_dict(), 'multilingual_model.pt')
        best_val_loss = val_loss

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Train:   0%|          | 0/2754 [00:00<?, ?it/s]

Eval:   0%|          | 0/57 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1 | Train loss: 9.586 | Val loss: 5.579


...

Eval:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 9 | Train loss: 3.448 | Val loss: 3.380


Train:   0%|          | 0/2754 [00:00<?, ?it/s]

Eval:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 10 | Train loss: 3.434 | Val loss: 3.377


### 7. Оптимизированная функция перевода

Изменения относительно семинарского кода прокомментированы в коде.

In [15]:
@torch.no_grad()
def translate_batch(texts, model, batch_size=256):
    model.eval()
    results = []

    bos_id = tokenizer_ru.token_to_id('[BOS]')
    eos_id = tokenizer_ru.token_to_id('[EOS]')

    for i in tqdm(range(0, len(texts), batch_size), desc='Translating'):
        batch_texts = texts[i:i + batch_size]
        encoded = [tokenizer_src.encode(t).ids[:max_len_src]
                   for t in batch_texts]

        # Выравниваем тензоры по самому длинному в текущем батче
        max_batch_len = max(len(seq) for seq in encoded)
        src_tensor = torch.full(
            (len(batch_texts),
             max_batch_len),
            PAD_IDX,
            dtype=torch.long,
            device=DEVICE)
        for j, seq in enumerate(encoded):
            src_tensor[j, :len(seq)] = torch.tensor(seq, dtype=torch.long)

        src_padding_mask = (src_tensor == PAD_IDX)
        output_ids = torch.full(
            (len(batch_texts), 1), bos_id, dtype=torch.long, device=DEVICE)

        # Тензор для отслеживания завершения каждой строки в батче
        finished = torch.zeros(
            len(batch_texts),
            dtype=torch.bool,
            device=DEVICE)

        for _ in range(max_len_ru):
            tgt_padding_mask = (output_ids == PAD_IDX)
            logits = model(
                src_tensor,
                output_ids,
                src_padding_mask,
                tgt_padding_mask)
            next_tokens = logits[:, -1, :].argmax(dim=-1)

            # Если генерация закончена, делаем паддинг, чтобы не
            # мусорить
            next_tokens = torch.where(
                finished, torch.tensor(
                    PAD_IDX, device=DEVICE), next_tokens)
            output_ids = torch.cat(
                [output_ids, next_tokens.unsqueeze(1)], dim=1)

            # Обновляем маску завершенности по токену EOS
            finished |= (next_tokens == eos_id)
            if finished.all():
                break

        for seq in output_ids.tolist():
            if eos_id in seq:
                seq = seq[1:seq.index(eos_id)]
            else:
                seq = seq[1:]
            seq = [token for token in seq if token != PAD_IDX]
            results.append(tokenizer_ru.decoder.decode(
                [tokenizer_ru.id_to_token(idx) for idx in seq]))

    return results

### 8. Оцениваемся на тестовых данных

In [16]:
def evaluate_language(src_texts, ref_texts, lang_name):
    preds = translate_batch(src_texts, model)

    bleu = sacrebleu.corpus_bleu(preds, [ref_texts]).score
    chrf = sacrebleu.corpus_chrf(preds, [ref_texts]).score

    print(f"--- {lang_name} -> RU ---")
    print(f"BLEU: {bleu:.2f} | chrF: {chrf:.2f}")
    return preds

In [17]:
print("Evaluating English...")
en_preds = evaluate_language(en_test, ru_test_en, "EN")

print("Evaluating German...")
de_preds = evaluate_language(de_test, ru_test_de, "DE")

print("Evaluating Spanish...")
es_preds = evaluate_language(es_test, ru_test_es, "ES")

Evaluating English...


Translating:   0%|          | 0/20 [00:00<?, ?it/s]

--- EN -> RU ---
BLEU: 30.29 | chrF: 48.44
Evaluating German...


Translating:   0%|          | 0/20 [00:00<?, ?it/s]

--- DE -> RU ---
BLEU: 23.18 | chrF: 40.65
Evaluating Spanish...


Translating:   0%|          | 0/20 [00:00<?, ?it/s]

--- ES -> RU ---
BLEU: 19.85 | chrF: 37.10


### 9. Смотрим на лучшие переводы для каждого языка

In [ ]:
def show_top_translations(src_texts, ref_texts, preds, lang_name, top_n=20):
    scored = []
    for src, ref, pred in zip(src_texts, ref_texts, preds):
        score = sacrebleu.sentence_chrf(pred, [ref]).score
        scored.append((score, src, ref, pred))

    scored.sort(key=lambda x: x[0], reverse=True)

    print(f"=== Top {top_n} translations for {lang_name} ===")
    for i, (score, src, ref, pred) in enumerate(scored[:top_n]):
        print(f"{i+1}. Score (chrF): {score:.2f}")
        print(f"SRC : {src}")
        print(f"REF : {ref}")
        print(f"PRED: {pred}")

In [21]:
show_top_translations(en_test, ru_test_en, en_preds, "EN")
show_top_translations(de_test, ru_test_de, de_preds, "DE")
show_top_translations(es_test, ru_test_es, es_preds, "ES")

=== Top 20 translations for EN ===
1. Score (chrF): 100.00
SRC : Just give me the phone.
REF : Просто дайте мне телефон.
PRED: Просто дайте мне телефон .
2. Score (chrF): 100.00
SRC : Get me a glass of milk.
REF : Принесите мне стакан молока.
PRED: Принесите мне стакан молока .
3. Score (chrF): 100.00
SRC : That's the fastest train in the world.
REF : Это самый быстрый поезд в мире.
PRED: Это самый быстрый поезд в мире .
4. Score (chrF): 100.00
SRC : He promised to call her.
REF : Он обещал ей позвонить.
PRED: Он обещал ей позвонить .
5. Score (chrF): 100.00
SRC : Tom lost hope.
REF : Том потерял надежду.
PRED: Том потерял надежду .
6. Score (chrF): 100.00
SRC : I'm not very happy, either.
REF : Я тоже не очень счастлив.
PRED: Я тоже не очень счастлив .
7. Score (chrF): 100.00
SRC : You never asked me for help.
REF : Ты никогда не просил меня о помощи.
PRED: Ты никогда не просил меня о помощи .
8. Score (chrF): 100.00
SRC : Tom said he wouldn't dance.
REF : Том сказал, что не будет тан

### 10. Тестируем в zero-shot-сетапе

In [20]:
print("Evaluating Dutch...")
nl_preds = evaluate_language(nl_test, ru_test_nl, "NL (Zero-shot)")
show_top_translations(nl_test,
                      ru_test_nl,
                      nl_preds,
                      "NL (Zero-shot)",
                      top_n=5)

print("\nEvaluating French...")
fr_preds = evaluate_language(fr_test, ru_test_fr, "FR (Zero-shot)")
show_top_translations(fr_test,
                      ru_test_fr,
                      fr_preds,
                      "FR (Zero-shot)",
                      top_n=5)

Evaluating Dutch...


Translating:   0%|          | 0/1 [00:00<?, ?it/s]

--- NL (Zero-shot) -> RU ---
BLEU: 0.70 | chrF: 10.02
=== Top 5 translations for NL (Zero-shot) ===
1. Score (chrF): 30.83
SRC : Tom had niet genoeg geld.
REF : У Тома было недостаточно денег.
PRED: У Тома была ненененененеомока .
2. Score (chrF): 30.41
SRC : Welkom in Japan.
REF : Добро пожаловать в Японию.
PRED: В Японии есть в Японии .
3. Score (chrF): 23.95
SRC : Toen ik klein was wandelde ik heel graag in de regen.
REF : Когда я был маленьким, я очень любил гулять под дождём.
PRED: В конце концов был маленький мойной .
4. Score (chrF): 23.25
SRC : De wind waait.
REF : Ветер дует.
PRED: В этом ветер .
5. Score (chrF): 22.70
SRC : Eenmaal op het station aangekomen, belde ik mijn vriend op.
REF : Приехав на станцию, я позвонил своему другу.
PRED: В конце концов он был хорошим начальником , на станции .

Evaluating French...


Translating:   0%|          | 0/2 [00:00<?, ?it/s]

--- FR (Zero-shot) -> RU ---
BLEU: 0.91 | chrF: 9.93
=== Top 5 translations for FR (Zero-shot) ===
1. Score (chrF): 56.51
SRC : Je suis allé au concert.
REF : Я пошёл на концерт.
PRED: Чем я на концерт .
2. Score (chrF): 50.79
SRC : Où sont tes parents ?
REF : Где твои родители?
PRED: Не трогай родители ?
3. Score (chrF): 42.41
SRC : Pour qui est ce message ?
REF : Для кого это сообщение?
PRED: У нас есть любимое сообщение ?
4. Score (chrF): 40.32
SRC : Je ne compare pas Tom à Mary.
REF : Я не сравниваю Тома с Мэри.
PRED: Чем Том взял Тома с Мэри .
5. Score (chrF): 37.20
SRC : Je sais que Tom veut faire bonne impression.
REF : Я знаю, что Том хочет произвести хорошее впечатление.
PRED: Чем Том , что Том не может впечатление .


### 11. Выводы

Судя по метрикам на целевых языках, модель обучилась очень даже неплохо — для обучения с нуля на ~700к примерах переводы коротких предложений вполне точные, а отсмотренные топовые переводы, как видно, идеально попадают в цель, имея метрики в 100%. При этом английский переводится лучше всех, а испанский — хуже всех. Скорее всего, это связано с размерами выборок для каждого языка, а также с морфологической простотой английского.

Межъязыковой перенос практически не работает, метрики очень низкие. При этом, в отличие от метрик для целевых языков, тут наблюдается разница больше чем в 10 раз между BLEU и chrF, что говорит о том, что модель угадывает отдельные полнофункциональные слова, но совершенно не справляется с переводом всего предложения, не понимает взаимосвязь между словами и синтаксис, что подтверждают переводы, на которые мы посмотрели. Оно, в прочем, и неудивительно: многие полнофункциональные слова в родственных языках более-менее общие или похожи, а вот синтаксис и служебные слова отличаются в максимальной степени.

Сама динамика обучения была очень хорошей, со стабильно снижающимся лоссом, без признаков переобучения или недообучения. Но мы, скорее всего, более-менее вовремя остановили обучение на 10 эпохах, так как в конце стало видно, что лосс уже почти не падает.

## Задание 2 (2 балла).
Прочитайте главу про машинный перевод у Журафски и Маннига - https://web.stanford.edu/~jurafsky/slp3/13.pdf
Ответьте своими словами в чем заключается техника back translation? Для чего она применяется и что позволяет получить? Опишите по шагам как ее применить к паре en->ru на данных из семинара. Сколько моделей понадобится? Сколько запусков обучения нужно будет сделать?

Ответ должен содержать как минимум 10 предложений.

Back translation — подход для аугментации данных, который помогает улучшить качество перевода, когда у нас не хватает параллельных текстов. Суть в том, чтобы использовать для обучения корпуса только для одного, целевого языка, сгенерировав для них искусственные исходные тексты на другом языке. Этот подход полезен для малоресурсных языков или для специфичных доменов, так как найти моноязычные корпуса сильно проще, чем двуязычные. В результате мы получаем синтетический набор данных, который можно добавить к основному параллельному корпусу, что должно повысить качество модели.

Чтобы применить этот подход к паре en-ru на наших данных, нам понадобится обучить две отдельные модели. Сначала мы берём какой-то небольшой параллельный корпус и обучаем на нём первую модель, которая будет переводить в обратном направлении — с русского на английский. Затем мы находим какой-нибудь большой корпус текстов только на русском языке и прогоняем его через эту первую модель. В результате мы получаем синтетические английские тексты, выровненный с естественными русскими.

Далее мы берём этот новый синтетический датасет и объединяем его с нашими изначальными параллельными данными. На этом объединённом корпусе мы запускаем второе обучение — теперь уже финальной модели, которая переводит с английского на русский. В итоге нам нужно провести два полноценных запуска обучения: один для создания вспомогательной модели обратного перевода и один для обучения целевой модели на расширенных данных.

По идеи мы получаем ситуацию, при которой качество текстов на исходном языке может быть не очень хорошим, но целевые переводы всё равно остаются естественными и грамотными, а это для нашей модели важнее — поэтому это по идеи лучше, чем, например, учиться на синтетических переводах для целевого языка.